# Protein Function Classifier in Google Colab

Train a PyTorch classifier on UniProt protein sequences. The notebook automatically uses a Colab GPU when available, uses a sequence-identity-aware split to reduce leakage, and trains with early stopping on validation loss.

This version simplifies the task to a narrower binary problem: oxidoreductases vs hydrolases.

In [ ]:
!pip -q install requests torch

import copy
import hashlib
import json
import random
import re
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import requests
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

UNIPROT_SEARCH_URL = 'https://rest.uniprot.org/uniprotkb/search'
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
AMINO_ACID_TO_INDEX = {amino_acid: index + 1 for index, amino_acid in enumerate(AA_ALPHABET)}
UNKNOWN_INDEX = len(AMINO_ACID_TO_INDEX) + 1
PAD_INDEX = 0

@dataclass(frozen=True)
class ProteinSample:
    sequence: str
    label: int

def make_session():
    session = requests.Session()
    session.headers.update({'User-Agent': 'sequenceview-protein-classifier/1.0', 'Accept': 'text/plain'})
    return session

def parse_fasta(text):
    sequences, current = [], []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if current:
                sequences.append(''.join(current))
                current = []
            continue
        current.append(line)
    if current:
        sequences.append(''.join(current))
    return sequences

def next_link(link_header):
    if not link_header:
        return None
    match = re.search(r'<([^>]+)>;\s*rel="next"', link_header)
    return match.group(1) if match else None

def download_uniprot_sequences(query, limit, batch_size=500, session=None):
    if limit <= 0:
        return []
    local_session = session or make_session()
    sequences = []
    url = UNIPROT_SEARCH_URL
    params = {'query': query, 'format': 'fasta', 'size': min(batch_size, limit)}
    while url and len(sequences) < limit:
        response = local_session.get(url, params=params, timeout=90)
        response.raise_for_status()
        sequences.extend(parse_fasta(response.text))
        url = next_link(response.headers.get('Link'))
        params = None
    return sequences[:limit]

def clean_sequence(sequence):
    allowed = set(AA_ALPHABET)
    return ''.join(character if character in allowed else 'X' for character in sequence.upper())

def stable_kmer_hash(kmer):
    digest = hashlib.sha1(kmer.encode('utf-8')).digest()
    return int.from_bytes(digest[:8], byteorder='big', signed=False)

def sequence_identity_key(sequence, k=5, signature_size=8):
    cleaned = clean_sequence(sequence)
    if len(cleaned) <= k:
        return ('short', len(cleaned), cleaned)
    kmer_hashes = sorted(
        stable_kmer_hash(cleaned[index:index + k])
        for index in range(len(cleaned) - k + 1)
    )
    length_bucket = len(cleaned) // 50
    return ('bucket', length_bucket, stable_kmer_hash(cleaned[:k]), stable_kmer_hash(cleaned[-k:]), tuple(kmer_hashes[:signature_size]))

def save_dataset(samples, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        for sample in samples:
            handle.write(json.dumps({'sequence': sample.sequence, 'label': sample.label}) + '\n')

def load_dataset(path):
    samples = []
    with path.open('r', encoding='utf-8') as handle:
        for raw_line in handle:
            record = json.loads(raw_line)
            samples.append(ProteinSample(str(record['sequence']), int(record['label'])))
    return samples

def build_labeled_dataset(oxidoreductase_count, hydrolase_count, cache_path=None, refresh=False):
    if cache_path and cache_path.exists() and not refresh:
        return load_dataset(cache_path)
    session = make_session()
    oxidoreductase_query = 'reviewed:true AND ec:1.* AND length:[50 TO 1000]'
    hydrolase_query = 'reviewed:true AND ec:3.* AND length:[50 TO 1000]'
    oxidoreductase_sequences = download_uniprot_sequences(oxidoreductase_query, oxidoreductase_count, session=session)
    hydrolase_sequences = download_uniprot_sequences(hydrolase_query, hydrolase_count, session=session)
    samples = [ProteinSample(clean_sequence(sequence), 1) for sequence in oxidoreductase_sequences]
    samples.extend(ProteinSample(clean_sequence(sequence), 0) for sequence in hydrolase_sequences)
    random.shuffle(samples)
    if cache_path:
        save_dataset(samples, cache_path)
    return samples

class ProteinSequenceDataset(Dataset):
    def __init__(self, samples, max_length):
        self.samples = list(samples)
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        return encode_sequence(sample.sequence, self.max_length), torch.tensor(sample.label, dtype=torch.long)

def encode_sequence(sequence, max_length):
    encoded = torch.full((max_length,), PAD_INDEX, dtype=torch.long)
    cleaned = clean_sequence(sequence)[:max_length]
    for position, character in enumerate(cleaned):
        encoded[position] = AMINO_ACID_TO_INDEX.get(character, UNKNOWN_INDEX)
    return encoded

def make_dataloader(samples, max_length, batch_size, shuffle):
    dataset = ProteinSequenceDataset(samples, max_length)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=torch.cuda.is_available())

class ProteinClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=192, dropout=0.25):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_INDEX)
        self.encoder = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 2),
        )

    def forward(self, inputs):
        mask = inputs != PAD_INDEX
        embeddings = self.embedding(inputs)
        outputs, _ = self.encoder(embeddings)
        outputs = outputs * mask.unsqueeze(-1)
        lengths = mask.sum(dim=1).clamp(min=1).unsqueeze(-1)
        mean_pool = outputs.sum(dim=1) / lengths
        max_pool = outputs.masked_fill(~mask.unsqueeze(-1), float('-inf')).max(dim=1).values
        max_pool = torch.where(torch.isfinite(max_pool), max_pool, torch.zeros_like(max_pool))
        features = torch.cat([mean_pool, max_pool], dim=1)
        return self.head(features)

def evaluate(model, data_loader, device):
    model.eval()
    loss_function = nn.CrossEntropyLoss(label_smoothing=0.05)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            logits = model(inputs)
            loss = loss_function(logits, labels)
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += batch_size
    return {
        'loss': total_loss / max(total_examples, 1),
        'accuracy': total_correct / max(total_examples, 1),
    }

def predict_sequence(model, sequence, max_length, device):
    model.eval()
    input_tensor = encode_sequence(sequence, max_length).unsqueeze(0).to(device)
    with torch.no_grad():
        probabilities = torch.softmax(model(input_tensor), dim=1)[0].cpu()
    prediction = int(probabilities.argmax().item())
    return {
        'label': prediction,
        'class_name': 'oxidoreductase' if prediction == 1 else 'hydrolase',
        'probabilities': {
            'hydrolase': float(probabilities[0].item()),
            'oxidoreductase': float(probabilities[1].item()),
        },
    }

def save_checkpoint(checkpoint_path, model, max_length, metadata):
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'state_dict': model.state_dict(),
        'max_length': max_length,
        'vocab_size': model.embedding.num_embeddings,
        'metadata': metadata,
    }, checkpoint_path)

def split_dataset(samples, train_fraction=0.8, validation_fraction=0.1, seed=13):
    if not 0 < train_fraction < 1:
        raise ValueError('train_fraction must be between 0 and 1')
    if not 0 <= validation_fraction < 1:
        raise ValueError('validation_fraction must be between 0 and 1')
    if train_fraction + validation_fraction >= 1:
        raise ValueError('train_fraction + validation_fraction must be less than 1')

    label_groups = {0: defaultdict(list), 1: defaultdict(list)}
    for sample in samples:
        label_groups[sample.label][sequence_identity_key(sample.sequence)].append(sample)

    rng = random.Random(seed)
    splits = {'train': [], 'validation': [], 'test': []}

    for grouped_samples in label_groups.values():
        cluster_groups = list(grouped_samples.values())
        rng.shuffle(cluster_groups)
        total_label_samples = sum(len(cluster) for cluster in cluster_groups)
        train_target = int(total_label_samples * train_fraction)
        validation_target = int(total_label_samples * validation_fraction)
        test_target = total_label_samples - train_target - validation_target
        targets = {
            'train': train_target,
            'validation': validation_target,
            'test': test_target,
        }
        current_counts = {'train': 0, 'validation': 0, 'test': 0}

        for cluster in cluster_groups:
            chosen_split = max(
                current_counts,
                key=lambda split_name: targets[split_name] - current_counts[split_name],
            )
            splits[chosen_split].extend(cluster)
            current_counts[chosen_split] += len(cluster)

    rng.shuffle(splits['train'])
    rng.shuffle(splits['validation'])
    rng.shuffle(splits['test'])
    return splits['train'], splits['validation'], splits['test']

def train_model(model, train_loader, validation_loader, device, epochs, learning_rate, patience=2):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    loss_function = nn.CrossEntropyLoss(label_smoothing=0.05)
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    best_validation_loss = float('inf')
    best_state_dict = copy.deepcopy(model.state_dict())
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_correct = 0.0
        running_examples = 0
        for inputs, labels in train_loader:
            inputs = inputs.to(device, non_blocking=use_amp)
            labels = labels.to(device, non_blocking=use_amp)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(inputs)
                loss = loss_function(logits, labels)
            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            running_correct += (torch.argmax(logits, dim=1) == labels).sum().item()
            running_examples += batch_size
        scheduler.step()
        train_metrics = {'loss': running_loss / max(running_examples, 1), 'accuracy': running_correct / max(running_examples, 1)}
        validation_metrics = evaluate(model, validation_loader, device)
        print(f"epoch {epoch + 1}/{epochs} train_loss={train_metrics['loss']:.4f} train_accuracy={train_metrics['accuracy']:.4f} validation_loss={validation_metrics['loss']:.4f} validation_accuracy={validation_metrics['accuracy']:.4f}")

        current_validation_loss = validation_metrics['loss']
        if current_validation_loss < best_validation_loss:
            best_validation_loss = current_validation_loss
            best_state_dict = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"early stopping at epoch {epoch + 1}; best_epoch={best_epoch} best_validation_loss={best_validation_loss:.4f}")
                break

    model.load_state_dict(best_state_dict)
    print(f"restored best model from epoch {best_epoch}")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    checkpoint_path = Path('/content/drive/MyDrive/protein_classifier.pt')
except Exception:
    checkpoint_path = Path('/content/protein_classifier.pt')

oxidoreductase_count = 50000
hydrolase_count = 50000
max_length = 768
batch_size = 64
epochs = 16
learning_rate = 3e-4
patience = 2
seed = 13
data_cache = Path('/content/uniprot_binary_dataset.jsonl')

target_examples = oxidoreductase_count + hydrolase_count
print(f"target_examples={target_examples}")

random.seed(seed)
torch.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
print(f"cuda_available={torch.cuda.is_available()}")

samples = build_labeled_dataset(oxidoreductase_count, hydrolase_count, cache_path=data_cache)
train_samples, validation_samples, test_samples = split_dataset(samples, seed=seed)
train_loader = make_dataloader(train_samples, max_length, batch_size, shuffle=True)
validation_loader = make_dataloader(validation_samples, max_length, batch_size, shuffle=False)
test_loader = make_dataloader(test_samples, max_length, batch_size, shuffle=False)
model = ProteinClassifier(vocab_size=len(AMINO_ACID_TO_INDEX) + 2)
print(f"parameters={sum(parameter.numel() for parameter in model.parameters()):,}")
train_model(model, train_loader, validation_loader, device=device, epochs=epochs, learning_rate=learning_rate, patience=patience)
test_metrics = evaluate(model, test_loader, device)
print(test_metrics)
save_checkpoint(checkpoint_path, model, max_length=max_length, metadata={
    'oxidoreductase_count': oxidoreductase_count,
    'hydrolase_count': hydrolase_count,
    'max_length': max_length,
    'batch_size': batch_size,
    'epochs': epochs,
    'learning_rate': learning_rate,
    'patience': patience,
    'seed': seed,
    'test_metrics': test_metrics,
})
print(checkpoint_path)
print(predict_sequence(model, 'MVLSPADKTNVKAA', max_length, device))